# Combined RAG Pipeline: Scientific Paper RAG

This notebook merges the three separate working notebooks (ingestion/chunking, the basic RAG pipeline, and evaluation) into a single end-to-end walkthrough. No code has been changed — cells are combined as-is, with markdown added to narrate each stage:

1. **Documents** — load the raw knowledge base
2. **Chunk** — split documents into overlapping chunks
3. **Embed and save in db** — embed chunks and persist them in a vector database
4. **Retrieve** — query the db and pull back related context
5. **Prompt the context** — feed that context to the LLM to answer the question

Plus two bonus sections: visualizing the vector store, and evaluating the pipeline.

Standard imports for document loading, chunking, embeddings, the vector store, and visualization, followed by an optional Hugging Face Hub login (needed if a local embedding model requires authentication). This cell also defines the shared Azure Foundry endpoint, chat model (`gpt-5.6-luna`), and embedding model (`text-embedding-3-large`) used throughout the notebook.

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    CharacterTextSplitter ,
    RecursiveCharacterTextSplitter ,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
)
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from huggingface_hub import login
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# NOTEBOOK_DIR assumes the notebook is run from the directory it lives in
# (Jupyter's default working directory), so the vector database ends up
# next to this .ipynb file rather than at a path relative to some other location.
NOTEBOOK_DIR = Path.cwd()

AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-6608-reso.services.ai.azure.com/openai/v1"
)

DEFAULT_MODEL = "gpt-5.6-luna"
EMBEDDING_MODEL = "text-embedding-3-large"

In [ ]:
load_dotenv()

_token = os.getenv("HF_TOKEN")
if _token:
    login(token=_token)


## 1. Documents
Load the raw knowledge base. First, do a quick scan of every `.md` file to see how much text we're dealing with in total. Then use LangChain's `DirectoryLoader` to properly load each file into a `Document` object, tagging each one with its `doc_type` (taken from its parent folder name) so we can trace it back later.

In [ ]:
# How many characters in all the documents?

knowledge_base_path = "../knowledge_base/**/*.md"


files = glob.glob(knowledge_base_path, recursive=True)
print(files)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("../knowledge_base/*")


documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")



In [ ]:
documents[0]


## 2. Chunk
Split each loaded document into smaller, overlapping chunks with `RecursiveCharacterTextSplitter`. Overlap helps preserve context across chunk boundaries, and smaller chunks make retrieval more precise than searching over whole documents.

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")


print(f"First chunk:\n\n{chunks[0]}")


In [ ]:
chunks[0]


In [ ]:

print(len(chunks))

Embed every chunk using Azure's `text-embedding-3-large` model (via `OpenAIEmbeddings` pointed at the Azure Foundry endpoint), then persist the vectors into a Chroma vector database on disk so they can be queried later without re-embedding. (The original local Hugging Face embedding model is left commented out for reference.)

In [ ]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings = OpenAIEmbeddings(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    model=EMBEDDING_MODEL,
    default_query={"api-version": "preview"},
)

db_name = str(NOTEBOOK_DIR / "vector_database")

In [ ]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_database = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=db_name
)

print(f"Vector Store created with {vector_database._collection.count()} documents")


In [ ]:
print(vector_database)

In [ ]:
collection = vector_database._collection
count = collection.count()

# Fetch one item, including its embedding vector
sample = collection.get(limit=1, include=["embeddings"])

# The embedding vector for that one item
sample_embedding = sample["embeddings"][0]

dimensions = len(sample_embedding)

print(f"Number of documents: {count}")
print(f"Number of dimensions: {dimensions}")


### Bonus: Visualize the vector store
Pull the raw embeddings, documents, and metadata back out of Chroma, then use t-SNE to project the high-dimensional vectors down to 2D and 3D so we can visually inspect how the different document types cluster together.

In [ ]:
# Prework

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]
colors = [
    ["blue", "green", "red", "orange"][
        ["products", "employees", "contracts", "company"].index(t)
    ]
    for t in doc_types
]


In [ ]:
import nbformat

print(nbformat.__version__)
print(nbformat.__file__)


In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()


In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()


In the standalone RAG pipeline notebook, we start fresh here: re-create the same Azure embedding function and reconnect to the already-persisted Chroma database built in the step above, plus confirm the Azure endpoint/model (`gpt-5.6-luna`) to use for generation.

In [ ]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os



# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings = OpenAIEmbeddings(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    model=EMBEDDING_MODEL,
    default_query={"api-version": "preview"},
)

db_name = str(NOTEBOOK_DIR / "vector_database")

vector_database = Chroma(persist_directory=db_name, embedding_function=embeddings)

In [ ]:
AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-6608-reso.services.ai.azure.com/openai/v1"
)

DEFAULT_MODEL = "gpt-5.6-luna"

## 4. Retrieve the query db and get related context
## 5. Prompt the context
Build a retriever (top-`k` similarity search over the vector store), a prompt template that injects the retrieved context alongside the user's question, and a `run_rag()` helper that ties it together: retrieve → format context → prompt the LLM → parse the answer.

In [ ]:

llm = ChatOpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    model=DEFAULT_MODEL,
    default_query={"api-version": "preview"},
)

retriever = vector_database.as_retriever(search_kwargs={"k": 5})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant answering questions about Insurellm. "
            "Use only the provided context to answer. If the answer isn't in "
            "the context, say you don't know.\n\n"
            "Context:\n{context}",
        ),
        ("human", "Question: {question}"),
    ]
)


def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('doc_type', 'unknown')} | {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in docs
    )


"""
RAG LCEL CHAIN
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Who is the CEO of Insurellm?")
print(answer)
"""


def run_rag(question: str) -> str:
    docs = retriever.invoke(question)

    context = format_docs(docs)

    messages = prompt.invoke({"context": context, "question": question})

    response = llm.invoke(messages)

    answer = StrOutputParser().invoke(response)

    return answer


answer = run_rag("Who studied at Manchester university")
print(answer)


### Bonus: Chat UI
Wrap `run_rag()` in a Gradio `ChatInterface` for interactive, conversational testing of the pipeline.

In [ ]:
def chat(message, history):
    return run_rag(message)

demo = gr.ChatInterface(
    fn=chat,
    title="Insurellm Knowledge Base Assistant",
    
)
demo.launch()


## Bonus: Evaluation
Load a set of test questions (each with a reference answer and expected keywords), then evaluate both **retrieval quality** and **answer quality** (accuracy, completeness, relevance) using the pipeline built above.

In [ ]:
import sys
sys.path.append("..")
from evaluation import test


In [ ]:
tests = test.load_tests()
print(tests)

In [ ]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


In [ ]:
from evaluation.eval import evaluate_retrieval, evaluate_answer


In [ ]:
evaluate_retrieval(example)


In [ ]:
eval, answer, chunks = evaluate_answer(example)


In [ ]:
eval


In [ ]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)
